### Tools

Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:

1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute

In [2]:
from dotenv import load_dotenv

load_dotenv()

True

In [26]:
from langchain.chat_models import init_chat_model

model = init_chat_model(model="groq:qwen/qwen3-32b")
# response = model.invoke("Hello, how are you?")

# print(response.content)

In [27]:
from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """Get the weather in a given location"""
    return f"It's sunny in {location}"

model_with_tools = model.bind_tools([get_weather])
    

In [28]:
response = model_with_tools.invoke("What's the weather in Tokyo?")

print(response)

content='' additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in Tokyo. Let me check the tools available. There\'s a function called get_weather that takes a location parameter. Since the user specified Tokyo, I need to call this function with "Tokyo" as the location. I\'ll make sure to format the tool call correctly within the XML tags as instructed.\n', 'tool_calls': [{'id': 'bj2mqtg87', 'function': {'arguments': '{"location":"Tokyo"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 95, 'prompt_tokens': 154, 'total_tokens': 249, 'completion_time': 0.150465353, 'completion_tokens_details': {'reasoning_tokens': 70}, 'prompt_time': 0.007546118, 'prompt_tokens_details': None, 'queue_time': 0.161364002, 'total_time': 0.158011471}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_2bfcc54d36', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} i

In [29]:
response.tool_calls

[{'name': 'get_weather',
  'args': {'location': 'Tokyo'},
  'id': 'bj2mqtg87',
  'type': 'tool_call'}]

### Tool execution loop

In [ ]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Tokyo?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)

print(final_response.text)


The weather in Tokyo is sunny right now! If you're there, it's a great day to enjoy outdoor activities. 😊
